In [1]:
from lstm_model import VolatilityLSTM
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from sklearn import preprocessing

# Data preparation

In [2]:
df = pd.read_csv("data/labelled_data.csv")
df["Date"] = pd.to_datetime(df["Date"])

df.drop(columns=["Unnamed: 0"], inplace=True)

train = df[(df["Date"] >= "2010-01-01") & (df["Date"] < "2022-01-01")]
val = df[(df["Date"] >= "2022-01-01") & (df["Date"] < "2024-01-01")]
test = df[(df["Date"] >= "2024-01-01")]

feature_scaler = preprocessing.StandardScaler()
target_scaler = preprocessing.StandardScaler()

feature_cols = ["log_returns", "squared_log_returns", "5_day_rolling_vol", "21_day_rolling_vol"]
target_col = "target_t"

trainX = feature_scaler.fit_transform(train[feature_cols])
trainY = target_scaler.fit_transform(train[target_col].values.reshape(-1, 1))

valX = feature_scaler.transform(val[feature_cols])
valY = target_scaler.transform(val[target_col].values.reshape(-1, 1))

testX = feature_scaler.transform(test[feature_cols])
testY = target_scaler.transform(test[target_col].values.reshape(-1, 1))

In [3]:
lstmXTraining, lstmYTraining = [], []

for i in range(0, len(trainX) - 21):
    lstmXTraining.append(trainX[i:i+21])
    lstmYTraining.append(trainY[i+20])

lstmXTraining = np.array(lstmXTraining)
lstmYTraining = np.array(lstmYTraining)

In [4]:
lstmXVal, lstmYVal = [], []

for i in range(0, len(valX) - 21):
    lstmXVal.append(valX[i:i+21])
    lstmYVal.append(valY[i+20])

lstmXVal = np.array(lstmXVal)
lstmYVal = np.array(lstmYVal)

In [5]:
lstmXTesting, lstmYTesting = [], []

for i in range(0, len(testX) - 21):
    lstmXTesting.append(testX[i:i+21])
    lstmYTesting.append(testY[i+20])

lstmXTesting = np.array(lstmXTesting)
lstmYTesting = np.array(lstmYTesting)

In [6]:
import torch

lstmXTraining = torch.from_numpy(lstmXTraining).float()
lstmYTraining = torch.from_numpy(lstmYTraining).float()

lstmXVal = torch.from_numpy(lstmXVal).float()
lstmYVal = torch.from_numpy(lstmYVal).float()

lstmXTesting = torch.FloatTensor(lstmXTesting)
lstmYTesting = torch.FloatTensor(lstmYTesting)

# Model tuning

In [7]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [8]:
lstmXTraining.shape

torch.Size([2979, 21, 4])

In [9]:
lstmYTraining.shape

torch.Size([2979, 1])

In [10]:
lstmYVal.shape

torch.Size([480, 1])

In [11]:
lstmYTesting.shape

torch.Size([475, 1])

## Mini-batching

In [12]:
training_dataset = TensorDataset(lstmXTraining, lstmYTraining)
validation_dataset = TensorDataset(lstmXVal, lstmYVal)

In [13]:
from torch import nn

In [14]:
loss_function = nn.MSELoss()

max_epochs = 50
#batchSize = 32
patience = 5
# delta = 0.001
delta = 0.01

In [15]:
import itertools 
# hyperparameters combinations 
# 2 * 3 * 3 * 2 = 36 combinations to try 
layers = [1, 2]
hidden_size = [32, 64, 128]
dropout = [0.1, 0.3, 0.5]
batch_sizes = [32, 64]

configuration_list = []

In [16]:
for layer, h_size, dropout_rate, batchSize in itertools.product(layers, hidden_size, dropout, batch_sizes):    
    torch.manual_seed(42)
    training_dataloader = DataLoader(training_dataset, batch_size=batchSize, shuffle=True)
    model = VolatilityLSTM(num_layers=layer, hidden_size=h_size, dropout_rate=dropout_rate)
    optimiser = torch.optim.Adam(model.parameters(), lr = 0.001)

    configuration = {}
    epoch = 0

    best_val_loss = float('inf')
    no_improvement_count = 0

    for i in range(max_epochs):
        model.train()
        for batch in training_dataloader:
            training_prediction = model(batch[0])
            training_loss = loss_function(training_prediction, batch[1])
            optimiser.zero_grad()
            training_loss.backward()
            optimiser.step()

        model.eval()
        epoch += 1
        with torch.no_grad():
            val_prediction = model(lstmXVal)
            val_loss = loss_function(val_prediction, lstmYVal)
            # print(f"Epoch {i} - Val Loss : {val_loss:.6f}")
            

            if val_loss < best_val_loss - delta:
                best_val_loss = val_loss
                no_improvement_count = 0
                # torch.save(model.state_dict(), "best_model2.pt")
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    print("Stopping early as no improvement has been observed.")
                    break

    configuration["layer"] = layer
    configuration["hidden_size"] = h_size
    configuration["dropout"] = dropout_rate
    configuration["batch_size"] = batchSize
    configuration["stopped_at_epoch"] = epoch
    configuration["best_val_loss"] = best_val_loss.item()
    configuration_list.append(configuration)

/Users/sid/Documents/COMP0162 Advanced ML/COMP0162-Advanced-Machine-Learning-Coursework/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Stopping early as no improvement has been observed.


/Users/sid/Documents/COMP0162 Advanced ML/COMP0162-Advanced-Machine-Learning-Coursework/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Stopping early as no improvement has been observed.


/Users/sid/Documents/COMP0162 Advanced ML/COMP0162-Advanced-Machine-Learning-Coursework/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.


/Users/sid/Documents/COMP0162 Advanced ML/COMP0162-Advanced-Machine-Learning-Coursework/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping early as no improvement has been observed.
Stopping ear

In [17]:
min(configuration_list, key=lambda x:x['best_val_loss'])

{'layer': 2,
 'hidden_size': 32,
 'dropout': 0.3,
 'batch_size': 64,
 'stopped_at_epoch': 7,
 'best_val_loss': 0.38803932070732117}

### Results visualisation

In [18]:
config_df = pd.DataFrame(configuration_list)
config_df.sort_values('best_val_loss')

,layer,hidden_size,dropout,batch_size,stopped_at_epoch,best_val_loss
21,2,32,0.3,64,7,0.388039
23,2,32,0.5,64,8,0.396349
19,2,32,0.1,64,7,0.401386
22,2,32,0.5,32,7,0.401838
9,1,64,0.3,64,7,0.407814
27,2,64,0.3,64,7,0.413649
4,1,32,0.5,32,7,0.414205
5,1,32,0.5,64,8,0.414407
11,1,64,0.5,64,8,0.414562
10,1,64,0.5,32,7,0.414865


### Rerunning the experiment with the 'best' parameters

In [19]:
batchSize = 64
layer = 2
h_size = 32
dropout_rate = 0.3

best_val_loss = float('inf')
no_improvement_count = 0
epoch = 0

In [20]:
torch.manual_seed(42)
model = VolatilityLSTM(num_layers=layer, hidden_size=h_size, dropout_rate=dropout_rate)
optimiser = torch.optim.Adam(model.parameters(), lr = 0.0001)

In [21]:
training_dataloader = DataLoader(training_dataset, batch_size=batchSize, shuffle=True)

In [22]:
for i in range(max_epochs):
    model.train()
    for batch in training_dataloader:
        training_prediction = model(batch[0])
        training_loss = loss_function(training_prediction, batch[1])
        optimiser.zero_grad()
        training_loss.backward()
        optimiser.step()

    model.eval()
    epoch += 1
    with torch.no_grad():
        val_prediction = model(lstmXVal)
        val_loss = loss_function(val_prediction, lstmYVal)
        # print(f"Epoch {i} - Val Loss : {val_loss:.6f}")
        

        if val_loss < best_val_loss - delta:
            best_val_loss = val_loss
            no_improvement_count = 0
            # torch.save(model.state_dict(), "best_model2.pt")
        else:
            no_improvement_count += 1
            if no_improvement_count >= patience:
                print("Stopping early as no improvement has been observed.")
                break

Stopping early as no improvement has been observed.


In [23]:
best_val_loss

tensor(0.3960)

In [24]:
epoch

22